# Session 6 — Geometric Transforms

**AI & Computer Vision Lecture Series**
**Phase 1 — Foundations  •  Week 3  •  Monday**
*Practical session — hands-on code*

---

## What we'll do today

Geometric transforms *move the pixels around* instead of changing their colors.
They're the foundation for:

- **Data augmentation** (rotating / scaling training images)
- **Alignment** (registering photos, stitching panoramas)
- **Correction** (straightening a photographed document, removing perspective)
- **Next session's milestone** — the *document scanner*

By the end of this notebook you will have used every OpenCV transform you need:

1. Translation
2. Rotation
3. Scaling / resizing
4. Flipping
5. **Affine** transform (`cv2.getAffineTransform` — 3 points)
6. **Perspective** transform / homography (`cv2.getPerspectiveTransform` — 4 points)
7. A tiny document-scanner preview

> Run each cell with **Shift + Enter** and play with the parameters —
> geometry becomes intuitive once you start dragging numbers around.


---
## 0 · Setup

We'll lean on OpenCV for the math and Matplotlib for display.
Reminder: OpenCV images are in **BGR** order, so we swap to RGB before plotting.


In [ ]:
# Uncomment if needed:
# !pip install opencv-python matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)


A small helper we'll reuse throughout.

In [ ]:
def show(images, titles=None, cmap=None, cols=None, figsize=None):
    """Display one or several images in a tidy grid."""
    if not isinstance(images, list):
        images = [images]
    n = len(images)
    cols = cols or n
    rows = int(np.ceil(n / cols))
    figsize = figsize or (5 * cols, 4 * rows)

    if titles is None:
        titles = [""] * n
    if isinstance(titles, str):
        titles = [titles]
    if not isinstance(cmap, list):
        cmap = [cmap] * n

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()

    for ax, img, t, cm in zip(axes, images, titles, cmap):
        if img.ndim == 3 and cm is None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap=cm or "gray")
        ax.set_title(t)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


---
## 1 · Test images

We'll use three images today:

1. **`dog.jpg`** — familiar from previous sessions
2. **`grace_hopper.jpg`** — a famous portrait (ships with Matplotlib, public domain)
3. **A synthetic "document"** — we'll build it from scratch, then photograph it
   at an angle, so we can un-warp it later with a perspective transform.


In [ ]:
dog     = cv2.imread("dog.jpg")
portrait = cv2.imread("grace_hopper.jpg")

assert dog is not None,     "dog.jpg not found next to this notebook"
assert portrait is not None, "grace_hopper.jpg not found next to this notebook"

print("dog     :", dog.shape)
print("portrait:", portrait.shape)

show([dog, portrait], titles=["dog.jpg", "grace_hopper.jpg"])


### 1.1 Building a synthetic "document"

We'll create a clean 600×800 white page with some text and a rectangle,
mimicking a receipt or ID card. Later we'll photograph it "at an angle"
and use a perspective transform to put it back flat.


In [ ]:
def make_document(w=600, h=800):
    doc = np.full((h, w, 3), 255, dtype=np.uint8)  # white page
    cv2.rectangle(doc, (30, 30), (w - 30, h - 30), (40, 40, 40), 3)
    cv2.putText(doc, "INVOICE",       (60, 120),  cv2.FONT_HERSHEY_SIMPLEX, 2.2, (20, 20, 20), 5)
    cv2.putText(doc, "ID: A-20260420", (60, 190),  cv2.FONT_HERSHEY_SIMPLEX, 1.0, (80, 80, 80), 2)
    cv2.line   (doc, (60, 230), (w - 60, 230),                        (120, 120, 120), 2)
    cv2.putText(doc, "Item          Qty    Price", (60, 290),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (40, 40, 40), 2)
    cv2.putText(doc, "Camera         1   $ 450",   (60, 360),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (40, 40, 40), 2)
    cv2.putText(doc, "Tripod         1   $  80",   (60, 420),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (40, 40, 40), 2)
    cv2.putText(doc, "Lens Filter    2   $  35",   (60, 480),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (40, 40, 40), 2)
    cv2.line   (doc, (60, 560), (w - 60, 560),                        (120, 120, 120), 2)
    cv2.putText(doc, "TOTAL   $ 565", (60, 630),  cv2.FONT_HERSHEY_SIMPLEX, 1.4, (20, 20, 180), 3)
    # corner markers so we can see orientation
    for cx, cy in [(30, 30), (w - 30, 30), (w - 30, h - 30), (30, h - 30)]:
        cv2.circle(doc, (cx, cy), 10, (0, 120, 200), -1)
    return doc

document = make_document()
show(document, titles=["Synthetic document (the 'ground truth' flat page)"])


---
## 2 · Translation — sliding the image

A translation moves every pixel by `(tx, ty)`. We describe it with a
**2×3 transformation matrix**:

$$
M = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \end{bmatrix}
$$

`cv2.warpAffine(img, M, (w, h))` does the actual pixel-moving.


In [ ]:
h, w = dog.shape[:2]

tx, ty = 100, 50  # shift right 100, down 50

M = np.float32([[1, 0, tx],
                [0, 1, ty]])

translated = cv2.warpAffine(dog, M, (w, h))

show([dog, translated],
     titles=["Original", f"Translated by ({tx}, {ty})"])


**Try it:** change `tx` to `-200` and `ty` to `0`. What part of the image goes where?

---
## 3 · Rotation

For rotation we use `cv2.getRotationMatrix2D(center, angle, scale)`:

- **center** — the pivot point (usually the image center)
- **angle** — in degrees, positive = counter-clockwise
- **scale** — zoom factor (1.0 = no zoom)


In [ ]:
h, w = dog.shape[:2]
center = (w // 2, h // 2)

angle = 30
M = cv2.getRotationMatrix2D(center, angle, 1.0)
rotated_cropped = cv2.warpAffine(dog, M, (w, h))

show([dog, rotated_cropped],
     titles=["Original", f"Rotated {angle}° (corners clipped)"])


### 3.1 Rotating *without* clipping the corners

When we rotate, the corners of the image fall outside the original canvas and get
chopped. To keep everything visible we compute a **bigger output canvas** and
adjust the translation component of the rotation matrix.


In [ ]:
def rotate_bound(img, angle):
    """Rotate `img` by `angle` degrees without cropping."""
    (h, w) = img.shape[:2]
    (cx, cy) = (w // 2, h // 2)

    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    # cos and sin from the rotation matrix
    cos = abs(M[0, 0]); sin = abs(M[0, 1])

    # new width / height that comfortably contains the rotated image
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))

    # shift the rotation so it's centered in the new canvas
    M[0, 2] += (new_w / 2) - cx
    M[1, 2] += (new_h / 2) - cy

    return cv2.warpAffine(img, M, (new_w, new_h), borderValue=(255, 255, 255))

rotated_full = rotate_bound(dog, 30)
show([dog, rotated_cropped, rotated_full],
     titles=["Original", "warpAffine (clipped)", "rotate_bound (full)"])


### 3.2 Quick grid of angles

In [ ]:
angles = [0, 45, 90, 135, 180, 270]
rotations = [rotate_bound(dog, a) for a in angles]
show(rotations, titles=[f"{a}°" for a in angles], cols=3, figsize=(15, 8))


---
## 4 · Scaling — making things bigger / smaller

Two ways:

1. **`cv2.resize()`** — easiest for simple scaling (with nice interpolation options)
2. **Scaling matrix + `warpAffine`** — equivalent, matches the affine pipeline

### 4.1 `cv2.resize`


In [ ]:
h, w = portrait.shape[:2]

# Scale to half size
half = cv2.resize(portrait, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)

# Exact target dimensions
fixed = cv2.resize(portrait, (400, 400), interpolation=cv2.INTER_LINEAR)

print("original:", portrait.shape)
print("half    :", half.shape)
print("fixed   :", fixed.shape)

show([portrait, half, fixed],
     titles=["Original", "×0.5 (INTER_AREA)", "400×400 (INTER_LINEAR)"])


**Rule of thumb:**
- **Shrinking** → `INTER_AREA` (best quality, no ringing)
- **Enlarging** → `INTER_CUBIC` or `INTER_LINEAR`
- **Default** (if unsure) → `INTER_LINEAR`

### 4.2 Scaling through a matrix (same result, different route)


In [ ]:
sx, sy = 1.5, 1.5
M = np.float32([[sx, 0, 0],
                [0, sy, 0]])
new_w, new_h = int(w * sx), int(h * sy)
scaled_matrix = cv2.warpAffine(portrait, M, (new_w, new_h))

show([portrait, scaled_matrix],
     titles=["Original", f"warpAffine scale ×{sx}"])


---
## 5 · Flipping (mirror)

`cv2.flip(img, flipCode)`:

| flipCode | Effect |
|----------|--------|
| 0        | Flip vertical (top ↔ bottom) |
| 1        | Flip horizontal (left ↔ right) |
| -1       | Flip both |


In [ ]:
flip_h = cv2.flip(dog, 1)
flip_v = cv2.flip(dog, 0)
flip_b = cv2.flip(dog, -1)

show([dog, flip_h, flip_v, flip_b],
     titles=["Original", "Horizontal (1)", "Vertical (0)", "Both (-1)"],
     cols=4)


---
## 6 · Affine Transform — the 3-point magic

An **affine** transformation preserves parallel lines.
Translations, rotations, scaling, and shearing are all affine.

`cv2.getAffineTransform(src_pts, dst_pts)` takes **three** point pairs
(before → after) and figures out the 2×3 matrix for you.


In [ ]:
img = portrait.copy()
h, w = img.shape[:2]

# Three reference points in the original image
src = np.float32([[0, 0],           # top-left
                  [w - 1, 0],       # top-right
                  [0, h - 1]])      # bottom-left

# Where we want those three points to land
dst = np.float32([[60, 80],         # top-left shifted inward & down
                  [w - 40, 20],     # top-right pushed up-right
                  [20, h - 70]])    # bottom-left shifted

M = cv2.getAffineTransform(src, dst)
print("Affine matrix M:")
print(M)

warped = cv2.warpAffine(img, M, (w, h), borderValue=(240, 240, 240))

# Draw the source and destination triangles so we can see what happened
vis_src = img.copy();    cv2.polylines(vis_src, [src.astype(int)], True, (0, 0, 255), 3)
vis_dst = warped.copy(); cv2.polylines(vis_dst, [dst.astype(int)], True, (0, 255, 0), 3)

show([vis_src, vis_dst],
     titles=["Source triangle (red)", "Warped — same triangle now green"])


### 6.1 Shearing with a matrix directly

A shear leaves one axis alone and slides the other proportionally.


In [ ]:
shear_x = 0.3   # horizontal shear
M_shear = np.float32([[1, shear_x, 0],
                      [0,       1, 0]])

sheared = cv2.warpAffine(portrait, M_shear,
                         (int(w + shear_x * h), h),
                         borderValue=(240, 240, 240))

show([portrait, sheared], titles=["Original", f"Shear x={shear_x}"])


---
## 7 · Perspective Transform — the 4-point magic

Affine keeps lines parallel. **Perspective transform** (aka **homography**) does
not — it can turn a trapezoid back into a rectangle, which is exactly what you
need to un-warp a photographed document.

`cv2.getPerspectiveTransform(src_pts, dst_pts)` uses **four** point pairs and
builds a 3×3 homography. We then apply it with `cv2.warpPerspective`.

### 7.1 First, let's simulate a photographed document


In [ ]:
# Take our synthetic 'flat' document and warp it as if we'd photographed it at
# an angle. The 4 destination corners are the distorted quadrilateral.
doc = document.copy()
dh, dw = doc.shape[:2]

# Original (flat) corners
src_flat = np.float32([[0, 0],
                       [dw - 1, 0],
                       [dw - 1, dh - 1],
                       [0, dh - 1]])

# "Photographed" corners — pretend we're looking at the page from the side.
dst_skewed = np.float32([[ 80,  50],
                         [540, 120],
                         [480, 720],
                         [ 50, 660]])

# Put the skewed document on a bigger background so it looks like a photo
canvas_w, canvas_h = 700, 800
canvas = np.full((canvas_h, canvas_w, 3), (30, 60, 90), dtype=np.uint8)

M_skew = cv2.getPerspectiveTransform(src_flat, dst_skewed)
warped_doc = cv2.warpPerspective(doc, M_skew, (canvas_w, canvas_h),
                                 borderMode=cv2.BORDER_TRANSPARENT,
                                 dst=canvas.copy())

# Highlight the 4 corner points so we can see them clearly
preview = warped_doc.copy()
for (x, y) in dst_skewed:
    cv2.circle(preview, (int(x), int(y)), 10, (0, 255, 255), -1)

show([document, preview],
     titles=["Original flat document", "\"Photograph\" of the document (skewed)"])


### 7.2 Reversing the perspective — un-warping

We want to go from the skewed photo back to a flat, rectified document.
So we use the **same** four points as source, and a nice clean rectangle as destination.


In [ ]:
OUT_W, OUT_H = 600, 800  # clean output size

src_photo = np.float32(dst_skewed)          # 4 corners we see in the photo
dst_clean = np.float32([[0, 0],
                        [OUT_W - 1, 0],
                        [OUT_W - 1, OUT_H - 1],
                        [0, OUT_H - 1]])

M_fix = cv2.getPerspectiveTransform(src_photo, dst_clean)
rectified = cv2.warpPerspective(warped_doc, M_fix, (OUT_W, OUT_H))

show([warped_doc, rectified],
     titles=["Skewed photo", "Rectified — back to a flat page!"])


**What you just did is the core of every document-scanner app.**
In next session we'll detect those 4 corners automatically with contours instead
of hard-coding them. Today we supplied them by hand so you can see the math clearly.

### 7.3 Visualizing the 4-point mapping

It's worth stopping to actually draw the source and destination quadrilaterals
next to each other — that's how the transform is specified.


In [ ]:
vis_src = warped_doc.copy()
cv2.polylines(vis_src, [src_photo.astype(int).reshape(-1, 1, 2)], True, (0, 255, 255), 3)
for i, (x, y) in enumerate(src_photo):
    cv2.putText(vis_src, str(i), (int(x) + 10, int(y) + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)

vis_dst = rectified.copy()
cv2.polylines(vis_dst, [dst_clean.astype(int).reshape(-1, 1, 2)], True, (0, 255, 255), 3)
for i, (x, y) in enumerate(dst_clean):
    cv2.putText(vis_dst, str(i), (int(x) + 10, int(y) + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)

show([vis_src, vis_dst], titles=["Source corners", "Destination corners"])


---
## 8 · Chaining transforms

Transformation matrices **compose by multiplication**.
E.g., "rotate then translate" = (Translation matrix) @ (Rotation matrix).

For affine transforms we usually just call `warpAffine` twice, but you can also
do it in a single shot by composing 3×3 homogeneous matrices:


In [ ]:
h, w = portrait.shape[:2]

# A pure rotation (as 3x3)
angle = 20
R = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
R3 = np.vstack([R, [0, 0, 1]])

# A pure translation
T3 = np.float32([[1, 0, 80],
                 [0, 1, 40],
                 [0, 0,  1]])

# Combined: first rotate, then translate
M3 = T3 @ R3
M2 = M3[:2, :]   # back to 2x3 for warpAffine

combined = cv2.warpAffine(portrait, M2, (w, h), borderValue=(240, 240, 240))
show([portrait, combined], titles=["Original", "Rotate 20° then translate (+80, +40)"])


---
## 9 · Interactive exploration *(optional)*

If `ipywidgets` is installed this gives you live sliders to tweak rotation
angle, scale, and translation on top of the portrait.


In [ ]:
try:
    from ipywidgets import interact, IntSlider, FloatSlider

    def play(angle=0, scale=1.0, tx=0, ty=0):
        h, w = portrait.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, scale)
        M[0, 2] += tx
        M[1, 2] += ty
        out = cv2.warpAffine(portrait, M, (w, h), borderValue=(240, 240, 240))
        plt.figure(figsize=(6, 6))
        plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
        plt.title(f"angle={angle}  scale={scale:.2f}  shift=({tx},{ty})")
        plt.axis("off"); plt.show()

    interact(play,
             angle=IntSlider(min=-180, max=180, step=5, value=0),
             scale=FloatSlider(min=0.3, max=2.0, step=0.1, value=1.0),
             tx=IntSlider(min=-200, max=200, step=10, value=0),
             ty=IntSlider(min=-200, max=200, step=10, value=0))
except ImportError:
    print("ipywidgets not installed — pip install ipywidgets  to enable sliders.")


---
## 10 · Mini challenge

Small exercises to sharpen your intuition:

1. **Rotate a photo 30° while keeping the whole image visible AND making the
   background red.** *(Hint: `borderValue=(0, 0, 255)`.)*
2. **Given a photo of `grace_hopper.jpg` tilted 15° to the right, un-tilt it.**
   Pick 3 reference points and use an affine transform.
3. **Chain three transforms on `dog.jpg`**: scale to half, rotate 45°, then
   translate by (+100, +0). Do it by composing 3×3 matrices (Section 8 style).
4. **Perspective challenge:** open a photo of a book on your desk, click on the
   4 corners with `plt.ginput(4)`, and un-warp it — a one-off document scanner.


In [ ]:
# Exercise 1 — rotation with red padding
rotated_red = rotate_bound(portrait, 30)
# (rotate_bound uses white — try editing it to take a border color argument)
show(rotated_red, titles=["Your turn: make the padding red"])


In [ ]:
# Exercise 3 — chained transforms starter code
h, w = dog.shape[:2]

# Step 1: scale
S = np.float32([[0.5, 0,   0],
                [0,   0.5, 0],
                [0,   0,   1]])

# Step 2: rotate 45° around (w/2, h/2) — build it as 3x3
R = cv2.getRotationMatrix2D((w // 2, h // 2), 45, 1.0)
R3 = np.vstack([R, [0, 0, 1]])

# Step 3: translate
T = np.float32([[1, 0, 100],
                [0, 1,   0],
                [0, 0,   1]])

# Compose and apply
M3 = T @ R3 @ S
M2 = M3[:2, :]
combined = cv2.warpAffine(dog, M2, (w, h), borderValue=(255, 255, 255))
show([dog, combined], titles=["Original", "Scale × Rotate × Translate"])


---
## 11 · Recap — functions to remember

| Task | Function |
|------|----------|
| Apply 2×3 affine matrix  | `cv2.warpAffine(img, M, (w, h))` |
| Apply 3×3 perspective    | `cv2.warpPerspective(img, M, (w, h))` |
| Rotate around a point    | `cv2.getRotationMatrix2D(center, angle, scale)` |
| Resize                   | `cv2.resize(img, dsize, fx, fy, interpolation)` |
| Mirror                   | `cv2.flip(img, flipCode)` |
| 3-point affine           | `cv2.getAffineTransform(src, dst)` |
| 4-point homography       | `cv2.getPerspectiveTransform(src, dst)` |

### Key intuitions

- **2×3 matrix** → affine → 6 degrees of freedom → preserves parallel lines.
- **3×3 matrix** → perspective → 8 degrees of freedom → fixes "tilted
  documents" and similar effects.
- Always think in terms of **point mappings**: "where do I want these
  corners to go?" — the matrix follows automatically.

### What's next — Session 7

We move to **Contours & shape analysis** (`findContours`, bounding boxes,
convex hull, moments). Combined with today's perspective transform, you'll be
able to build a **document scanner** in Session 8.

---

*End of Session 6 — great work!*
